# Самостоятельная работа: задачи 16–23
По рабочей тетради 3-4. Цапков М.А. ЭПМО-01-26

Язык — Python 3. Используется только стандартная библиотека.


## 16. Поиск повторяющихся идентификаторов

**Условие.** N = 100 000. Проверить наличие повторов; тест: [105, 207, 318, 412, 207, 519].

### 1. Алгоритм и структуры данных

Просматриваем ID и сохраняем встреченные значения в `set`. При первом повторе возвращаем `True`.

### 2. Обоснование выбора

Множество заменяет сравнение с каждым последующим элементом одной проверкой наличия.

### 3. Вычислительная сложность и масштабирование

Исходный алгоритм: O(N²) времени, O(1) памяти. В худшем случае N(N−1)/2 = 4 999 950 000 сравнений. Новый: в среднем O(N) времени и O(N) памяти; худшее время при коллизиях O(N²).

### 4. Программная реализация


In [1]:
def has_duplicates(ids):
    seen = set()
    for identifier in ids:
        if identifier in seen:
            return True
        seen.add(identifier)
    return False


### 5. Проверка и результаты

Проверяем ответы через `assert`. Фактические результаты приведены под ячейкой.


In [2]:
values = [105, 207, 318, 412, 207, 519]
assert has_duplicates(values)
assert not has_duplicates([1, 2, 3])
assert not has_duplicates([])
assert not has_duplicates([1])
assert has_duplicates([1, 1])
assert not has_duplicates(range(100_000))
print("Есть повтор:", has_duplicates(values))
print("Попарных сравнений:", 100_000 * 99_999 // 2)


Есть повтор: True
Попарных сравнений: 4999950000


### 6.* Вывод и возможная оптимизация

Хеш-множество сокращает ожидаемое время до линейного за счет памяти. При ограниченной памяти можно сортировать ID и сравнивать соседей за O(N log N).


## 17. Сортировка большого потока

**Условие.** Выбрать сортировку с гарантией O(N log N) для N = 1 000 000; тест: [38, 12, 57, 4, 29, 16, 41, 8].

### 1. Алгоритм и структуры данных

Сортировка слиянием: делим массив пополам, сортируем части рекурсивно и сливаем двумя указателями.

### 2. Обоснование выбора

В отличие от пузырьковой сортировки время O(N log N) гарантировано для любого порядка данных.

### 3. Вычислительная сложность и масштабирование

Время O(N log N), дополнительная память O(N), глубина рекурсии O(log N). Для миллиона элементов N log₂N ≈ 19 931 569, тогда как N² = 10¹². Это оценки порядка работы, а не точные числа инструкций.

### 4. Программная реализация


In [3]:
def merge_sort(values):
    if len(values) <= 1:
        return values[:]
    middle = len(values) // 2
    left = merge_sort(values[:middle])
    right = merge_sort(values[middle:])
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result


### 5. Проверка и результаты

Проверяем ответы через `assert`. Фактические результаты приведены под ячейкой.


In [4]:
values = [38, 12, 57, 4, 29, 16, 41, 8]
print("Отсортировано:", merge_sort(values))
assert merge_sort(values) == [4, 8, 12, 16, 29, 38, 41, 57]
for sample in [[], [1], [2, 1], [3, 3, -1], list(range(30)), list(range(30, 0, -1))]:
    assert merge_sort(sample) == sorted(sample)


Отсортировано: [4, 8, 12, 16, 29, 38, 41, 57]


### 6.* Вывод и возможная оптимизация

Слияние дает предсказуемое время, но требует буфера. В прикладном Python разумно использовать встроенный sorted; для данных больше памяти — внешнюю сортировку слиянием.


## 18. Поиск в высоконагруженном каталоге

**Условие.** N = 1 000 000, Q = 50 000/с: 90 % точных и 10 % диапазонных запросов.

### 1. Алгоритм и структуры данных

Запись — (ID, показатель). Строим словарь по ID и отсортированный список (показатель, ID). Бинарным поиском находим левую границу диапазона, затем читаем подходящие записи.

### 2. Обоснование выбора

Словарь обслуживает 45 000 точных запросов/с, упорядоченный индекс — 5 000 диапазонных. Полный просмотр требует до NQ = 5 × 10¹⁰ проверок/с.

### 3. Вычислительная сложность и масштабирование

Построение O(N log N), память O(N). Точный поиск в среднем O(1); диапазон O(log N + K), где K — размер ответа. Вставка в упорядоченный список O(N).

### 4. Программная реализация


In [5]:
from bisect import bisect_left


def build_catalog(records):
    by_id = dict(records)
    if len(by_id) != len(records):
        raise ValueError("Повтор ID")
    index = sorted((value, identifier) for identifier, value in records)
    keys = [value for value, identifier in index]
    return by_id, index, keys


def range_search(catalog, low, high):
    by_id, index, keys = catalog
    result = []
    i = bisect_left(keys, low)
    while i < len(index) and index[i][0] <= high:
        value, identifier = index[i]
        result.append((identifier, value))
        i += 1
    return result


### 5. Проверка и результаты

Проверяем ответы через `assert`. Фактические результаты приведены под ячейкой.


In [6]:
catalog = build_catalog([(1, 30), (2, 10), (3, 20), (4, 20)])
assert catalog[0].get(3) == 20
assert catalog[0].get(99) is None
assert range_search(catalog, 10, 20) == [(2, 10), (3, 20), (4, 20)]
assert range_search(catalog, 40, 50) == []
assert range_search(catalog, 30, 10) == []
assert range_search(build_catalog([]), 0, 10) == []
print("ID 3:", catalog[0].get(3))
print("Диапазон [10, 20]:", range_search(catalog, 10, 20))
print("Проверок при полном просмотре/с:", 1_000_000 * 50_000)


ID 3: 20
Диапазон [10, 20]: [(2, 10), (3, 20), (4, 20)]
Проверок при полном просмотре/с: 50000000000


### 6.* Вывод и возможная оптимизация

Индексы выгодны при многократных запросах. При частых обновлениях вместо списка нужен индекс на сбалансированном дереве; большие диапазоны все равно требуют O(K) на выдачу.


## 19. Классическое умножение и Карацуба

**Условие.** Сравнить пять уровней рекурсии и вычислить 1234 · 5678 с одним уровнем Карацубы.

### 1. Алгоритм и структуры данных

Разбиваем X = aB + b, Y = cB + d. Вычисляем z₂ = ac, z₀ = bd, z₁ = (a+b)(c+d)−z₂−z₀. Ответ z₂B² + z₁B + z₀. Для одного уровня B = 100.

### 2. Обоснование выбора

Карацуба использует три рекурсивных умножения вместо четырех; сложения дешевле больших умножений.

### 3. Вычислительная сложность и масштабирование

Для n цифр классическое разбиение T(n)=4T(n/2)+O(n), то есть O(n²); Карацуба T(n)=3T(n/2)+O(n), то есть O(n^log₂3) ≈ O(n^1.585). На пятом уровне 4⁵=1024 и 3⁵=243 листовых умножения; отношение ≈ 4.214. Речь об одном уровне дерева, а не сумме всех вызовов. Рекурсивная реализация требует O(n) дополнительной памяти при последовательном вычислении ветвей.

### 4. Программная реализация


In [7]:
def karatsuba(x, y):
    sign = -1 if (x < 0) != (y < 0) else 1
    x, y = abs(x), abs(y)
    if x < 10 or y < 10:
        return sign * x * y
    half = max(len(str(x)), len(str(y))) // 2
    base = 10 ** half
    a, b = divmod(x, base)
    c, d = divmod(y, base)
    z2 = karatsuba(a, c)
    z0 = karatsuba(b, d)
    z1 = karatsuba(a + b, c + d) - z2 - z0
    return sign * (z2 * base ** 2 + z1 * base + z0)


### 5. Проверка и результаты

Проверяем ответы через `assert`. Фактические результаты приведены под ячейкой.


In [8]:
a, b, c, d = 12, 34, 56, 78
z2, z0 = a * c, b * d
z1 = (a + b) * (c + d) - z2 - z0
result = z2 * 100 ** 2 + z1 * 100 + z0
print("ac, bd, средний коэффициент:", z2, z0, z1)
print("Произведение:", result)
print("Пятый уровень:", 4 ** 5, 3 ** 5, "отношение:", round(4 ** 5 / 3 ** 5, 3))
assert result == karatsuba(1234, 5678) == 7_006_652
for x, y in [(0, 123), (-1234, 5678), (-99, -101), (123456789, 987654321), (12, 3456789)]:
    assert karatsuba(x, y) == x * y


ac, bd, средний коэффициент: 672 2652 2840
Произведение: 7006652
Пятый уровень: 1024 243 отношение: 4.214


### 6.* Вывод и возможная оптимизация

На малых числах рекурсия дороже обычного умножения. Практическая оптимизация — переходить к встроенному умножению ниже выбранного порога.


## 20. Комплексная оптимизация запросов

**Условие.** N = 8 000 000. Найти повторы, сумму, максимум и число отрицательных значений; этап 16 с разделить на 8 модулей с расходами 0,8 с.

### 1. Алгоритм и структуры данных

За один проход по (ID, значение) обновляем множество ID и три агрегата. Обработку продолжаем после повтора, чтобы агрегаты учитывали все записи. Параллельный этап оцениваем по заданным временам.

### 2. Обоснование выбора

Один проход уменьшает число обращений к данным; множество устраняет квадратичный поиск. Для пустого пакета максимум равен None.

### 3. Вычислительная сложность и масштабирование

Последовательная часть: в среднем O(N) времени, O(N) памяти. T₈=16/8+0.8=2.8 с; ускорение S=16/2.8≈5.714; эффективность E=S/8≈71.43 %. Это показатели отдельного параллельного этапа. Если последовательная часть занимает t, общее ускорение равно (t+16)/(t+2.8).

### 4. Программная реализация


In [9]:
def aggregate(records):
    seen = set()
    duplicate = False
    total = negatives = 0
    maximum = None
    for identifier, value in records:
        if identifier in seen:
            duplicate = True
        seen.add(identifier)
        total += value
        negatives += value < 0
        if maximum is None or value > maximum:
            maximum = value
    return duplicate, total, maximum, negatives


def parallel_metrics(seconds, workers, overhead):
    elapsed = seconds / workers + overhead
    speedup = seconds / elapsed
    return elapsed, speedup, speedup / workers


### 5. Проверка и результаты

Проверяем ответы через `assert`. Фактические результаты приведены под ячейкой.


In [10]:
records = [(1, 10), (2, -3), (1, 7), (3, -2)]
assert aggregate(records) == (True, 12, 10, 2)
assert aggregate([]) == (False, 0, None, 0)
assert aggregate([(1, -5), (2, -2)]) == (False, -7, -2, 2)
time, speedup, efficiency = parallel_metrics(16, 8, 0.8)
assert abs(time - 2.8) < 1e-10
print("Повтор, сумма, максимум, отрицательные:", aggregate(records))
print(f"Время: {time:.1f} с; ускорение: {speedup:.3f}; эффективность: {efficiency:.2%}")


Повтор, сумма, максимум, отрицательные: (True, 12, 10, 2)
Время: 2.8 с; ускорение: 5.714; эффективность: 71.43%


### 6.* Вывод и возможная оптимизация

Ускорение оплачивается памятью множества и накладными расходами. Реальная параллельная реализация зависит от вида обработки; здесь рассчитывается заданная модель, запуск восьми процессов для учебных данных не нужен.


## 21. Оптимизация сервиса событий

**Условие.** Миллион событий (ID пользователя, значение): найти повторы, агрегаты и отсортировать значения.

### 1. Алгоритм и структуры данных

В одном проходе собираем значения и вычисляем агрегаты функцией aggregate из задачи 20; затем используем merge_sort из задачи 17. Для объединения проходов реализуем сбор значений вместе с агрегатами.

### 2. Обоснование выбора

Хеш-множество, один проход и сортировка слиянием заменяют три узких места, указанных в условии.

### 3. Вычислительная сложность и масштабирование

Исходное решение O(N²) времени: попарный поиск и пузырьковая сортировка; агрегаты O(N). Новое в среднем O(N log N), память O(N). Для N=10⁶ порядок сортировки ≈2×10⁷ вместо 10¹².

### 4. Программная реализация


In [11]:
def process_events(events):
    seen = set()
    duplicate = False
    values = []
    total = negatives = 0
    maximum = None
    for identifier, value in events:
        duplicate |= identifier in seen
        seen.add(identifier)
        values.append(value)
        total += value
        negatives += value < 0
        if maximum is None or value > maximum:
            maximum = value
    return duplicate, total, maximum, negatives, merge_sort(values)


### 5. Проверка и результаты

Проверяем ответы через `assert`. Фактические результаты приведены под ячейкой.


In [12]:
result = process_events([(1, 10), (2, -3), (1, 7), (3, -2)])
assert result == (True, 12, 10, 2, [-3, -2, 7, 10])
assert process_events([]) == (False, 0, None, 0, [])
print("Повтор, сумма, максимум, отрицательные, сортировка:")
print(result)


Повтор, сумма, максимум, отрицательные, сортировка:
(True, 12, 10, 2, [-3, -2, 7, 10])


### 6.* Вывод и возможная оптимизация

Асимптотику теперь определяет сортировка. Для высокой нагрузки можно обрабатывать пакеты, а при недостатке памяти сортировать внешним слиянием.


## 22. Индексация каталога и кэш

**Условие.** 2 млн записей, 20 000 запросов/с: 85 % точных, 15 % диапазонных. Нужны кэш и перестроение после пакетного обновления.

### 1. Алгоритм и структуры данных

Используем индексы задачи 18 и ограниченный LRU-кэш результатов. Ключ кэша содержит тип и параметры запроса. При пакетной замене данных перестраиваем индексы и очищаем кэш.

### 2. Обоснование выбора

Словарь обслуживает 17 000 точных запросов/с; упорядоченный индекс — 3 000 диапазонных. Кэш полезен для повторяющихся диапазонов. Полный просмотр потребовал бы до 4×10¹⁰ проверок/с.

### 3. Вычислительная сложность и масштабирование

Построение O(N log N), точный запрос в среднем O(1), диапазон при промахе O(log N+K). Попадание в кэш O(1) без учета чтения ответа. Память индексов O(N), кэша O(CKmax) для C ответов. Очистка кэша зависит от суммарного размера хранимых ответов.

### 4. Программная реализация


In [13]:
from collections import OrderedDict


class CachedCatalog:
    def __init__(self, records, capacity=3):
        if capacity < 1:
            raise ValueError("Емкость должна быть положительной")
        self.capacity = capacity
        self.cache = OrderedDict()
        self.rebuild(records)

    def rebuild(self, records):
        self.catalog = build_catalog(records)
        self.cache.clear()

    def query(self, kind, *args):
        key = (kind,) + args
        if key in self.cache:
            self.cache.move_to_end(key)
            return self.cache[key]
        if kind == "id":
            result = self.catalog[0].get(args[0])
        elif kind == "range":
            result = tuple(range_search(self.catalog, *args))
        else:
            raise ValueError("Неизвестный тип запроса")
        self.cache[key] = result
        if len(self.cache) > self.capacity:
            self.cache.popitem(last=False)
        return result


### 5. Проверка и результаты

Проверяем ответы через `assert`. Фактические результаты приведены под ячейкой.


In [14]:
service = CachedCatalog([(1, 30), (2, 10), (3, 20)], capacity=2)
assert service.query("id", 1) == 30
assert service.query("range", 10, 20) == ((2, 10), (3, 20))
assert service.query("id", 1) == 30
service.query("id", 99)
assert list(service.cache) == [("id", 1), ("id", 99)]
service.rebuild([(1, 40), (2, 15)])
assert not service.cache
assert service.query("id", 1) == 40
assert service.query("range", 10, 20) == ((2, 15),)
print("После обновления ID 1:", service.query("id", 1))
print("Диапазон [10, 20]:", service.query("range", 10, 20))


После обновления ID 1: 40
Диапазон [10, 20]: ((2, 15),)


### 6.* Вывод и возможная оптимизация

Ограниченный кэш не растет бесконечно по числу записей, но большие ответы требуют ограничения по памяти. Под нагрузкой можно строить новый индекс отдельно и атомарно заменять его вместе с версией кэша.


## 23. Вычислительный модуль

**Условие.** Карацуба для 1234 и 5678, анализ и сортировка массива результатов, расчет параллелизма: 16 с, 8 модулей, расходы 0,8 с.

### 1. Алгоритм и структуры данных

Повторно используем karatsuba из задачи 19, merge_sort из задачи 17 и parallel_metrics из задачи 20. После умножения собираем массив результатов, находим агрегаты и сортируем.

### 2. Обоснование выбора

Карацуба и слияние используют «разделяй и властвуй». Независимые умножения можно распределять между модулями, затем объединять результаты.

### 3. Вычислительная сложность и масштабирование

Для M пар n-значных чисел: O(M n^log₂3 + M log M) времени в модели операций над цифрами, память O(M+n). Анализ массива O(M). По заданной модели время этапа 2.8 с, ускорение 5.714, эффективность 71.43 %. Сортировка и объединение должны входить в измеренные расходы либо учитываться отдельно.

### 4. Программная реализация


In [15]:
def compute_batch(pairs):
    products = [karatsuba(x, y) for x, y in pairs]
    total = sum(products)
    maximum = max(products, default=None)
    return total, maximum, merge_sort(products)


### 5. Проверка и результаты

Проверяем ответы через `assert`. Фактические результаты приведены под ячейкой.


In [16]:
result = compute_batch([(1234, 5678), (12, 12), (-2, 5)])
assert result == (7_006_786, 7_006_652, [-10, 144, 7_006_652])
assert compute_batch([]) == (0, None, [])
print("Сумма, максимум, отсортированные произведения:", result)
time, speedup, efficiency = parallel_metrics(16, 8, 0.8)
print(f"Параллельный этап: {time:.1f} с, S={speedup:.3f}, E={efficiency:.2%}")


Сумма, максимум, отсортированные произведения: (7006786, 7006652, [-10, 144, 7006652])
Параллельный этап: 2.8 с, S=5.714, E=71.43%


### 6.* Вывод и возможная оптимизация

Основные алгоритмы проверены на небольшом пакете. Для реального ускорения нужны достаточно крупные независимые задачи и измерение стоимости передачи больших чисел между процессами.
